In [ ]:
import pandas as pd
import duckdb
import os
import requests
from tqdm import tqdm 
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from master_map import *

In [ ]:

ustat_path = "data/bronze/understat_historical.parquet"
fpl_path = "data/bronze/FPL_historical.parquet"
umatch = "data/bronze/EPL_match_summary.parquet"

curr = "data/bronze/current_fpl_season.parquet"

fpl_cleaned = "data/silver/fpl_cleaned"
current_fpl = "data/bronze/current_fpl_season.parquet"
fpl_merged = "data/silver/FPL_merged"
current_fixture = "data/bronze/FPL_fixture.parquet"

### BRONZE LAYER

In [ ]:
## Past season leagues standings (current season not included)
seasons = ['2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']
table = []

for season in seasons:
    url = f"https://sdms.planetsport.com/api/football/competition/EPL/ranking/{season}?division_type=total&order_by=points&order=desc"
    
    try:
        r = requests.get(url)
        r.raise_for_status()
        league_data = r.json()

        if 'data' in league_data:
            formatted_season = f"{season[:5]}{season[-2:]}"
            for item in league_data['data']:
                table.append({
                    'season': formatted_season, # Vital to keep track of which year this is!
                    'rank' : item['rank'],
                    'team' : item['team_name'],
                    'points' : float(item['points']), # Convert string to float for SQL
                    'won': int(item['won']),
                    'drawn': int(item['drawn']),
                    'lost': int(item['lost']),
                    'gf': int(item['gf']),
                    'ga': int(item['ga']),
                    'gd': int(item['goal_difference']),
                })
        print(f"Success: Fetched league table for {season}")
    except Exception as e:
        print(f"Error fetching {season}: {e}")

    time.sleep(2)

historical_standings = pd.DataFrame(table)
historical_standings.to_parquet("data/bronze/historical_standings")
from master_map import patch_work
patch_work("data/bronze/historical_standings")

In [ ]:
## Here, I am backfilling current seasons data as I started this project around GW 23ish and storinng it in a seperate table 
# which will be merged later onto the historical in the silver layer. And after a few transofrmation, a master table will be available in the Gold layer.
from ingestion_scripts.weekly_sync import *
run_fpl_pipeline()

#### The reason why I am ingesting data only from 2016-17 to 2024-25:
    - I want to add the column/feature 'code' (an unique id) for all the players from Chris Musson's FPL_id_map because vaastav has not added code in his dataset (in the merged gws)
    - And clean the data to sync with my weekly ingest and current season backfill maps the code from the bootstrapstatic while ingesting, so we don't need to worry about that later

In [ ]:
# Ingesting and cleaning historical data
filepath = os.path.join(os.getcwd(), "data/bronze/FPL_historical.parquet")

In [ ]:

fpl_seasons = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

base_url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/"

def fetch_fpl_season(season):
    url = f"{base_url}{season}/gws/merged_gw.csv"
    try:
        df = pd.read_csv(url, encoding='latin1', low_memory=False)
        df['season'] = season
        exclude_cols = ["id", "GW", "completed_passes", "dribbles", "ea_index", 
                        "errors_leading_to_goal", "errors_leading_to_goal_attempt", 
                        "fouls", "key_passes", "offside", "open_play_crosses", 
                        "big_chances_created", "big_chances_missed", "mng_win", 
                        "mng_loss", "mng_draw", "mng_clean_sheets", "mng_goals_scored", 
                        "mng_underdog_draw", "mng_underdog_win", "kickoff_time_formatted", 
                        "loaned_in", "loaned_out", "penalties_conceded", "tackled", 
                        "target_missed", "winning_goals"]
        
        df.drop(columns=exclude_cols, inplace=True, errors='ignore')
        if 'position' in df.columns:
            df = df[df['position'] != 'AM']
            
        df['kickoff_time'] = pd.to_datetime(df['kickoff_time']).dt.strftime('%Y-%m-%d %H:%M:%S')
        print(f"Successfully downloaded: {season}")
        return df
    
    except Exception as e:
        print(f"Error ingesting {season}: {e}")
        return None

print(f"Starting parallel download for {len(fpl_seasons)} seasons...")

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_fpl_season, fpl_seasons))

all_seasons_data = [df for df in results if df is not None]

if all_seasons_data:
    final_df = pd.concat(all_seasons_data, axis=0, ignore_index=True)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    final_df.to_parquet(filepath, index=False, compression='snappy')
    
    print(f"\n Hitorical data ingest, complete!")
    print(f"Total rows: {len(final_df)}")
else:
    print("No data was recovered. Check your URLs or internet connection.")

In [ ]:
print(len(pd.read_parquet("data/bronze/FPL_historical.parquet")))

In [ ]:
## Code mapping historical data as its a unique id and the current weekly ingest takes this into account
# Getting the fpl_id map from github
id_filepath = r"C:\Programing\Projects\Fantasy Premier League\data\bronze\FPL_idmap.parquet"
elementxcode_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/FPL/"
seasons = ["16-17", "17-18", "18-19", "19-20", "20-21", "21-22", "22-23", "23-24", "24-25"]

def fetch_season(season):
    url = f"{elementxcode_url}{season}.csv"
    try:
        df = pd.read_csv(url)
        if season in df.columns:
            df = df.rename(columns={season: 'element'})
        df['season'] = f"20{season}"
        return df
    except Exception as e:
        print(f"Could not retrieve {season}: {e}")
        return None

with ThreadPoolExecutor(max_workers=5) as executor:
    id_results = list(executor.map(fetch_season, seasons))

id_data = pd.concat([df for df in id_results if df is not None], ignore_index=True).to_parquet(id_filepath)

duckdb.execute(f"""
    CREATE OR REPLACE TABLE memory_join AS 
    SELECT 
        df2.code::VARCHAR AS code, 
        df1.*
    FROM read_parquet('{filepath}') AS df1
    LEFT JOIN read_parquet('{id_filepath}') AS df2
        ON df1.element = df2.element
        AND df1.season = df2.season
""")

duckdb.execute(f"COPY memory_join TO '{filepath}' (FORMAT 'PARQUET')")

duckdb.execute("DROP TABLE memory_join")

print(f"Successfully joined 'code' and updated: FPL_historical")


### SILVER LAYER

In [ ]:
## Columns to be analyzed 
df = pd.read_parquet(fpl_path)
df.columns[df.isnull().any()].tolist()   

In [ ]:

# ## Combine understat id to test_fpl
# from master_map import bankai
# df = bankai(test_path, ustat_path = None)

# ## There is this one GHOST game which was postponed but still in the dataset fixture - 27, ARS vs MAN CITY
# duckdb.sql(f"""
#     COPY (
#         SELECT * FROM read_parquet('{test_path}')
#         WHERE NOT (fixture = 275 AND season = '2019-20' AND kickoff_time = '2020-03-11 19:30:00')
#     ) TO '{test_path}' (FORMAT PARQUET);
# """)

##### Calcualting past seasons fixture table

In [ ]:
## Combine understat id to test_fpl
from master_map import bankai
fpl_df = bankai(fpl_path, ustat_path = None)

## There is this one GHOST game which was postponed but still in the dataset fixture - 27, ARS vs MAN CITY

duckdb.sql(f""" 
    WITH fplxuid AS (
        SELECT * FROM fpl_df
        WHERE NOT (fixture = 275 
           AND season = '2019-20' 
           AND kickoff_time = '2020-03-11 19:30:00')  
    ), 
    ultimate_ustat AS(
        SELECT 
            player_id, 
            m.match_id, 
            position as u_position, 
            u.team_name,
            home_team, away_team, 
            h_goals, a_goals, 
            m.datetime, 
            m.season
        FROM read_parquet('{ustat_path}') as u
        LEFT JOIN read_parquet('{umatch}') as m
        ON u.match_id = m.match_id
    ),
    score_map AS (
        SELECT 
            DISTINCT 
            temp.match_id,
            test.fixture,
            test.round,
            temp.home_team,
            temp.away_team,
            temp.h_goals,
            temp.a_goals,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM fplxuid) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM ultimate_ustat) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1  
        ),
    hist_bps AS (
        SELECT 
           fixture, 
           round, 
           sum(bps) filter (where was_home = true) as h_bps,
           sum(bps) filter (where was_home = false) as a_bps,
           kickoff_time, 
           season
        FROM fplxuid
        GROUP BY kickoff_time, round, season, fixture
        order by season, round, kickoff_time
    )
        Select h.fixture as fixture_id, h.round, home_team, away_team, h_bps, a_bps, h_goals as h_score, a_goals as a_score, h.kickoff_time, h.season
        from hist_bps as h
        left join score_map as s
        on s.fixture = h.fixture
        and s.season = h.season
        order by h.kickoff_time, h.season, h.fixture 
""").write_parquet("data/bronze/FPL_historical_fixture.parquet")


##### Imputing historical FPL for null h/a_scores, opponent team mapping and player position clean up

In [ ]:
duckdb.sql(f""" 
    WITH fplxuid AS (
        SELECT * FROM fpl_df
        WHERE NOT (fixture = 275 
           AND season = '2019-20' 
           AND kickoff_time = '2020-03-11 19:30:00')  
    ),
    ultimate_ustat AS(
        SELECT 
            player_id, 
            m.match_id, 
            position as u_position, 
            u.team_name,
            home_team, away_team, 
            h_goals, a_goals, 
            h_xg, a_xg,
            m.datetime, 
            m.season
        FROM read_parquet('{ustat_path}') as u
        LEFT JOIN read_parquet('{umatch}') as m
        ON u.match_id = m.match_id
    ),
    score_map AS (
        SELECT 
            DISTINCT 
            temp.match_id,
            test.fixture,
            temp.away_team, 
            temp.home_team,
            temp.a_goals,
            temp.h_goals,
            temp.h_xg,
            temp.a_xg,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM fplxuid) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM ultimate_ustat) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1  
    )
        SELECT 
            f.* exclude (u_id)
            replace(
            coalesce(team, case when was_home = true then home_team else away_team end) as team,
            coalesce(team_a_score, a_goals) as team_a_score, 
            coalesce(team_h_score, h_goals) as team_h_score,
            CASE 
                WHEN was_home = true then away_team else home_team
                END AS opponent_team
            )
        FROM fplxuid as f
        LEFT JOIN score_map as s
        ON f.fixture = s.fixture
        AND f.season = s.season
""").write_parquet("data/silver/FPL_cleaned")
   # code, name, position, team, was_home, home_team, away_team, team_a_score, team_h_score, expected_goals_conceded, a_goals, h_goals, a_xg, h_xg

In [ ]:
## Cleaning player positions using understat 
from master_map import shikai, patch_work
df = shikai(fpl_path="data/silver/FPL_cleaned", ustat_path= ustat_path)

duckdb.sql(""" 
        SELECT * EXCLUDE(u_position)
           REPLACE(
           COALESCE(position, u_position) as position
           )
        FROM df
    """).write_parquet("data/silver/FPL_cleaned")

patch_work("data/silver/FPL_cleaned")

#### Feature Engineering

In [ ]:
df = pd.read_parquet("data/silver/FPL_cleaned")
#display(df.info())
df.columns[df.isnull().any()].tolist()   

## Run everythin below this after every gameweek update and increase GW in the ML part

##### Merging current and past player stats

In [ ]:
from master_map import patch_work

duckdb.sql(f"""
        SELECT * FROM read_parquet("data/silver/FPL_cleaned")
        UNION BY NAME
        SELECT * FROM read_parquet("data/bronze/current_fpl_season.parquet")
""").write_parquet("data/silver/FPL_merged")


patch_work("data/silver/FPL_merged")

##### Merging current and past fixtures

In [ ]:
duckdb.sql("""
        SELECT * FROM read_parquet("data/bronze/FPL_historical_fixture.parquet")
        UNION BY NAME
        SELECT * FROM read_parquet("data/bronze/FPL_fixture.parquet")
""").write_parquet("data/silver/Fixture_merged")


patch_work("data/silver/Fixture_merged") ## Team mapping

In [ ]:
fplxuid = bankai("data\silver\FPL_merged", None)
fplxustat = bankai("data\silver\FPL_merged", "data/bronze/understat_historical.parquet")

duckdb.sql(""" 
    WITH ultimate_ustat AS(
        SELECT 
            player_id,
            m.match_id, 
            u.team_name,
            CASE 
                WHEN h_a = 'h' then a_xg
                ELSE h_xg
                END as xGc,
            m.datetime, 
            m.season
        FROM read_parquet("data/bronze/understat_historical.parquet") as u
        LEFT JOIN read_parquet("data/bronze/EPL_match_summary.parquet") as m
        ON u.match_id = m.match_id
    ),
    team_xGc AS (
        SELECT DISTINCT
            temp.match_id,
            test.fixture,
            test.team,
            temp.xGc,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM fplxuid) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM ultimate_ustat) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1 
    ),
    team_bps AS (
        SELECT * from (SELECT fixture_id, round, home_team as team, h_bps as bps_conceded, season, kickoff_time FROM read_parquet("data/silver/Fixture_merged"))
        UNION ALL
        SELECT * from (SELECT fixture_id, round, away_team as team, a_bps as bps_conceded, season, kickoff_time FROM read_parquet("data/silver/Fixture_merged"))   
        ),
    team_metrics AS (
        SELECT tb.*, tx.xGc
        FROM team_bps as tb
        LEFT JOIN team_xGc as tx
        ON tb.kickoff_time = tx.kickoff_time
        AND tb.fixture_id = tx.fixture
        AND tb.team = tx.team
    ),
    rolling_team_metrics AS (
        SELECT *,
        AVG(bps_conceded) OVER (
            PARTITION BY team, season 
            ORDER BY kickoff_time 
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS opp_rolling_bps,
        AVG(xGc) OVER (
            PARTITION BY team, season 
            ORDER BY kickoff_time 
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS opp_rolling_xGc
        FROM team_metrics
    ),
    cold_rolling_team_metrics AS (
    SELECT *,
    COALESCE(
        AVG(bps_conceded) OVER (
            PARTITION BY team, season 
            ORDER BY kickoff_time 
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING),
        AVG(bps_conceded) OVER (
            PARTITION BY season 
            ORDER BY kickoff_time 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    ) AS opp_rolling_bps,
    
    COALESCE(
        AVG(xGc) OVER (
            PARTITION BY team, season 
            ORDER BY kickoff_time 
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING),
        AVG(xGc) OVER (
            PARTITION BY season 
            ORDER BY kickoff_time 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    ) AS opp_rolling_xGc
    FROM team_metrics
)                           
        SELECT 
            code, name ,position, f.round, value / 10 as price, 
            AVG(minutes) OVER (
            PARTITION BY code, f.season 
            ORDER BY f.kickoff_time 
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) as player_rolling_minutes,
            AVG(creativity) OVER (
            PARTITION BY code, f.season 
            ORDER BY f.kickoff_time 
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) as player_rolling_creativity,
            AVG(threat) OVER (
            PARTITION BY code, f.season 
            ORDER BY f.kickoff_time 
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) as player_rolling_threat,
            AVG(bps) OVER (
            PARTITION BY code, f.season 
            ORDER BY f.kickoff_time 
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS player_rolling_bps, 
            f.team, f.opponent_team, was_home, opp_rolling_bps, opp_rolling_xGc, f.season, f.kickoff_time,
            total_points
        FROM fplxustat as f
        LEFT JOIN cold_rolling_team_metrics as r
        ON f.opponent_team = r.team
        AND f.kickoff_time = r.kickoff_time
""").write_parquet("data/gold/FPL_transformed")
patch_work('data/gold/FPL_transformed')

In [ ]:
patch_work(('data/bronze/historical_standings'))

In [ ]:
duckdb.sql("""SELECT *,
            points/(won+drawn+lost) as ppg
            from read_parquet('data/bronze/historical_standings')
            where season = '2024-25'
           """)

##### Training/Testing the model

In [ ]:
import matplotlib
df = pd.read_parquet("data/gold/FPL_transformed")
df = df.sort_values(['code', 'season', 'round'])
features_to_lag = ['player_rolling_bps', 'player_rolling_creativity', 'player_rolling_threat', 'player_rolling_minutes','opp_rolling_bps', 'opp_rolling_xGc', 'price']
for feat in features_to_lag:
    df[f'prev_{feat}'] = df.groupby('code')[feat].shift(1)

In [ ]:
ppg_map = {
    'Liverpool': 2.21, 'Arsenal': 1.95, 'Man City': 1.87, 'Chelsea': 1.82,
    'Newcastle': 1.74, 'Aston Villa': 1.74, "Nott'm Forest": 1.71, 'Brighton': 1.61,
    'Bournemouth': 1.47, 'Brentford': 1.47, 'Fulham': 1.42, 'Crystal Palace': 1.39,
    'Everton': 1.26, 'West Ham': 1.13, 'Man Utd': 1.11, 'Wolves': 1.11, 'Spurs': 1.00,
    # Promoted Baseline
    'Sunderland': 1.00, 'Leeds': 1.00, 'Burnley': 1.00 
}

# Add the feature to your main dataframe
# df['team_prev_ppg'] = df['team'].map(ppg_map)
# df['opp_prev_ppg'] = df['opponent_team'].map(ppg_map)

In [ ]:
df[['prev_price', 'was_home', 'prev_player_rolling_bps', 'prev_player_rolling_creativity', 'prev_player_rolling_threat', 'prev_player_rolling_minutes', 'prev_opp_rolling_bps', 'prev_opp_rolling_xGc', 'total_points']].corr()

In [ ]:
# 2. Simple Encoding
df['was_home'] = df['was_home'].astype(int)
df['pos'] = df['position']
df = pd.get_dummies(df, columns=['position']) # Turns 'position' into 4 columns

# 3. Temporal Split (The Pro Move)
train = df[df['season'] < '2024-25']
test = df[df['season'] == '2024-25']

# 4. Define Features (X) and Target (y)
features = ['prev_price', 'was_home', 'prev_player_rolling_bps', 'prev_player_rolling_creativity', 'prev_player_rolling_threat', 'prev_player_rolling_minutes', 'prev_opp_rolling_bps', 'prev_opp_rolling_xGc', 'position_DEF', 'position_MID', 'position_FWD']
X_train = train[features]
y_train = train['total_points']

X_test = test[features]
y_test = test['total_points']

# 5. Train the "Brain"
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
model.fit(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(model)
plt.show()

In [ ]:
GW_31 = df[(df['season'] == '2025-26') & (df['round'] == 31)].copy()

In [ ]:
# 1. Prep the GW31 features (X_gw31)
# Make sure the column order matches your X_train exactly!
X_gw31 = GW_31[features] 

# 2. Run the Brain
GW_31['predicted_points'] = model.predict(X_gw31)

# 3. View the Top 5 Picks for the week
GW_31[['name', 'team', 'total_points','predicted_points']].sort_values(by='predicted_points', ascending=False).head(10)

In [ ]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(GW_31['total_points'], GW_31['predicted_points'])
print(f"Average Error per Player: {mae:.2f} points")

In [ ]:
# Saving the model
model.save_model("models/fpl_xgboost_v1.json")

##### Predicting for upcoming week

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
from ingestion_scripts.fpl_scrapper import *
n = 32

boot_data = get_fpl_metadata()
# Unpack both, but use '_' for the first value
_, team_map = get_player_mappings(boot_data)

future_fixtures = get_upcoming_fixtures(team_map, n)

latest_player_stats = duckdb.sql("""
            SELECT code, name, position, round, price, player_rolling_minutes, player_rolling_creativity, player_rolling_threat, player_rolling_bps, team
            FROM (
                SELECT *,
                ROW_NUMBER() OVER (PARTITION BY code ORDER BY kickoff_time DESC) AS rn
                FROM read_parquet("data/gold/FPL_transformed")
                ) AS ranked
                WHERE rn = 1 
                AND season = '2025-26'
""").to_df()


latest_team_stats = duckdb.sql("""
            SELECT opponent_team, opp_rolling_bps, opp_rolling_xGc
            FROM (
                SELECT *,
                ROW_NUMBER() OVER (PARTITION BY opponent_team ORDER BY kickoff_time DESC) AS rn
                FROM read_parquet("data/gold/FPL_transformed")
                ) AS ranked
                WHERE rn = 1 
                AND season = '2025-26'
""").to_df()

pred_gw = duckdb.sql("""
                SELECT code, name, position, 
                     price as prev_price, player_rolling_minutes as prev_player_rolling_minutes , player_rolling_creativity as prev_player_rolling_creativity, player_rolling_threat as prev_player_rolling_threat, player_rolling_bps as prev_player_rolling_bps, 
                     team, opponent_team, ff.was_home, ff.round, 
                     opp_rolling_bps as prev_opp_rolling_bps, opp_rolling_xGc as prev_opp_rolling_xGc 
                FROM latest_player_stats as lp
                INNER JOIN future_fixtures as ff
                ON lp.team = ff.team_name
                LEFT JOIN latest_team_stats as lt
                ON ff.opponent_name = lt.opponent_team
""").to_df()

In [ ]:

# prep data
pred_gw['was_home'] = pred_gw['was_home'].astype(int)
pred_gw['pos'] = pred_gw['position']
pred_gw = pd.get_dummies(pred_gw, columns=['position']) 

# pred_gw['team_prev_ppg'] = pred_gw['team'].map(ppg_map)
# pred_gw['opp_prev_ppg'] = pred_gw['opponent_team'].map(ppg_map)

features = ['prev_price', 'was_home', 'prev_player_rolling_bps', 'prev_player_rolling_creativity', 'prev_player_rolling_threat', 'prev_player_rolling_minutes', 'prev_opp_rolling_bps', 'prev_opp_rolling_xGc','position_DEF', 'position_MID', 'position_FWD']

loaded_model = xgb.XGBRegressor() 
loaded_model.load_model("models/fpl_xgboost_v1.json")

pred_gw['predicted_points'] = loaded_model.predict(pred_gw[features])
pred_gw.sort_values(['predicted_points'], ascending= False)

In [ ]:
pred_gw_copy = pred_gw[pred_gw['prev_player_rolling_minutes'] >= 60].copy()

In [ ]:
pred_gw_copy.head(20).sort_values(['predicted_points'], ascending= False)

In [ ]:
# Best 15
import pulp


prob = pulp.LpProblem("FPL_Optimization", pulp.LpMaximize)


player_vars = pulp.LpVariable.dicts("player", pred_gw_copy.index, cat='Binary')
captain_vars = pulp.LpVariable.dicts("captain", pred_gw_copy.index, cat='Binary')


prob += pulp.lpSum([pred_gw_copy.loc[i, 'predicted_points'] * player_vars[i] for i in pred_gw_copy.index]) + \
        pulp.lpSum([pred_gw_copy.loc[i, 'predicted_points'] * captain_vars[i] for i in pred_gw_copy.index])


prob += pulp.lpSum([pred_gw_copy.loc[i, 'prev_price'] * player_vars[i] for i in pred_gw_copy.index]) <= 98.10

prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index]) == 15
prob += pulp.lpSum([captain_vars[i] for i in pred_gw_copy.index]) == 1

for i in pred_gw_copy.index:
    prob += captain_vars[i] <= player_vars[i]

for team in pred_gw_copy['team'].unique():
    prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index if pred_gw_copy.loc[i, 'team'] == team]) <= 3

# Note: You can adjust these for your preferred formation
prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index if pred_gw_copy.loc[i, 'pos'] == 'GKP']) == 2
prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index if pred_gw_copy.loc[i, 'pos'] == 'DEF']) == 5
prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index if pred_gw_copy.loc[i, 'pos'] == 'MID']) == 5
prob += pulp.lpSum([player_vars[i] for i in pred_gw_copy.index if pred_gw_copy.loc[i, 'pos'] == 'FWD']) == 3

# Solve it!
prob.solve()

# 8. Extract the AI-Picked Squad
selected_indices = [i for i in pred_gw_copy.index if player_vars[i].varValue == 1]
lineup = pred_gw_copy.loc[selected_indices]
lineup['is_captain'] = [captain_vars[i].varValue for i in selected_indices]

display(lineup[['name', 'team', 'pos', 'predicted_points', 'is_captain', 'prev_price', 'prev_player_rolling_bps', 'prev_opp_rolling_bps', 'prev_opp_rolling_xGc']].sort_values('pos'))
total_predicted = lineup['predicted_points'].sum() + lineup[lineup['is_captain'] == 1]['predicted_points'].sum()
total_budget = lineup['prev_price'].sum()
print(f"\nTotal Expected Points (Inc. Captain): {total_predicted:.2f}")
print(f"\nTotal budget: {total_budget:.2f}")